In [ ]:
# daily.py
# daily update：用 Finnhub quote 抓 OHLC，Yahoo 抓 volume
import requests, datetime, pandas as pd, time
from database import get_engine
from get_symbols import get_active_symbols
from sqlalchemy import text

FINNHUB_TOKEN = "d65fkg9r01qlmj2v19r0d65fkg9r01qlmj2v19rg"

def get_yahoo_volume_for_date(symbol: str, target_date: datetime.date):
    """抓 Yahoo 當日 volume"""
    yesterday = target_date - datetime.timedelta(days=1)
    period1 = int(time.mktime(yesterday.timetuple()))
    period2 = int(time.mktime((target_date + datetime.timedelta(days=1)).timetuple()))
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
    params = {
        "period1": period1,
        "period2": period2,
        "interval": "1d",
        "events": "history",
        "includeAdjustedClose": "true"
    }
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        if not data.get("chart") or not data["chart"].get("result"):
            return None
        result = data['chart']['result'][0]
        quote = result['indicators']['quote'][0]
        vols = quote.get('volume', [])
        if not vols:
            return None
        return int(vols[-1]) if vols[-1] is not None else None
    except Exception as e:
        print(f"yahoo volume error {symbol}: {e}")
        return None

def insert_daily_ignore_duplicates(df, engine):
    """INSERT ... ON CONFLICT DO NOTHING"""
    if df is None or df.empty:
        return 0
    cols = list(df.columns)
    insert_sql = f"""
    INSERT INTO stock_data ({', '.join(cols)})
    VALUES ({', '.join([f":{c}" for c in cols])})
    ON CONFLICT ON CONSTRAINT uq_symbol_datatype_ts DO NOTHING
    """
    with engine.begin() as conn:
        for row in df.to_dict(orient="records"):
            conn.execute(text(insert_sql), row)
    return len(df)

def update_daily_data():
    """每日收市後執行 daily update（data_type='D'）"""
    today = datetime.date.today()
    # Weekend skip
    if today.weekday() in (5, 6):  # Saturday=5, Sunday=6
        print("Weekend, skip daily update")
        return 0

    symbols = get_active_symbols()
    engine = get_engine()
    count = 0
    for s in symbols:
        try:
            url = f"https://finnhub.io/api/v1/quote"
            resp = requests.get(url, params={"symbol": s, "token": FINNHUB_TOKEN}, timeout=10)
            resp.raise_for_status()
            q = resp.json()
            vol = get_yahoo_volume_for_date(s, today)

            # Null check: 如果冇 close 價，skip
            if q.get("c") is None:
                print(f"{s} daily empty, skip")
                continue

            df = pd.DataFrame([{
                "symbol": s,
                "data_type": "D",
                "ts": today,
                "open": q.get("o"),
                "high": q.get("h"),
                "low": q.get("l"),
                "close": q.get("c"),
                "volume": vol
            }])
            df = df.dropna(subset=["open","high","low","close"])
            if df.empty:
                print(f"{s} daily empty after dropna, skip")
                continue

            n = insert_daily_ignore_duplicates(df, engine)
            count += n
        except Exception as e:
            print(f"update_daily error {s}: {e}")
    print(f"Daily update done, inserted {count} rows")
    return count

if __name__ == "__main__":
    update_daily_data()
